<style>
:root {
  --ink: #17202a;
  --muted: #64748b;
  --paper: #fbf7ef;
  --card: #fffaf0;
  --line: #e7dcc9;
  --navy: #11263f;
  --teal: #0f766e;
  --amber: #b45309;
  --rose: #be123c;
}
body, .jp-Notebook { background: var(--paper); color: var(--ink); }
.jp-Cell, .cell { margin-bottom: 1.35rem; }
h1, h2, h3 { color: var(--navy); letter-spacing: -0.03em; }
h1 { font-size: 2.4rem; margin-bottom: .35rem; }
h2 { font-size: 1.55rem; margin-top: 2rem; }
p, li { font-size: 1.02rem; line-height: 1.65; }
a { color: var(--teal); font-weight: 700; }
.jano-hero { background: linear-gradient(135deg, #fff7ed 0%, #f0fdfa 54%, #e0f2fe 100%); border: 1px solid var(--line); border-radius: 28px; padding: 2rem; box-shadow: 0 22px 70px rgba(17, 38, 63, .10); }
.jano-eyebrow { color: var(--amber); text-transform: uppercase; font-size: .75rem; font-weight: 800; letter-spacing: .16em; }
.jano-lead { color: #334155; max-width: 900px; font-size: 1.08rem; }
.jano-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(170px, 1fr)); gap: 14px; margin: 1rem 0 1.25rem; }
.jano-card { background: var(--card); border: 1px solid var(--line); border-radius: 18px; padding: 1rem; box-shadow: 0 10px 30px rgba(17, 38, 63, .07); }
.jano-card strong { display: block; color: var(--muted); font-size: .72rem; text-transform: uppercase; letter-spacing: .12em; margin-bottom: .35rem; }
.jano-card span { color: var(--navy); font-size: 1.35rem; font-weight: 800; }
.jano-note { border-left: 5px solid var(--teal); background: #ecfeff; border-radius: 16px; padding: 1rem 1.2rem; color: #164e63; }
.jano-warning { border-left-color: var(--amber); background: #fffbeb; color: #78350f; }
.jano-section { background: rgba(255,255,255,.58); border: 1px solid var(--line); border-radius: 24px; padding: 1.25rem 1.35rem; margin: 1.2rem 0; }
.jano-table table, table.dataframe { border-collapse: collapse; width: 100%; font-size: .9rem; background: white; border-radius: 16px; overflow: hidden; box-shadow: 0 12px 35px rgba(17, 38, 63, .08); }
.jano-table th, .jano-table td, table.dataframe th, table.dataframe td { border-bottom: 1px solid #edf2f7; padding: .68rem .72rem; text-align: right; }
.jano-table th, table.dataframe th { background: #11263f; color: #f8fafc; font-weight: 700; }
.jano-table tr:nth-child(even) td, table.dataframe tr:nth-child(even) td { background: #f8fafc; }
.output_area, .jp-OutputArea-output { margin-top: .65rem; }
pre, code { font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace; }
pre.jano-code { background: #0f172a; color: #d1fae5; border-radius: 18px; padding: 1rem; overflow-x: auto; font-size: .83rem; line-height: 1.55; border: 1px solid #1e293b; }
details.jano-code-wrap { margin: .65rem 0; }
details.jano-code-wrap summary { cursor: pointer; color: var(--teal); font-weight: 800; font-size: .9rem; }
.jano-pill { display: inline-flex; align-items: center; gap: .35rem; border-radius: 999px; background: #ccfbf1; color: #134e4a; padding: .32rem .7rem; font-weight: 800; font-size: .82rem; margin: .2rem .25rem .2rem 0; }
.jano-pill.amber { background: #fef3c7; color: #92400e; }
.jano-pill.rose { background: #ffe4e6; color: #9f1239; }
</style>
<div class="jano-hero">
  <div class="jano-eyebrow">Real dataset walkthrough</div>
  <h1>Jano on BTS Airline On-Time Performance</h1>
  <p class="jano-lead">This notebook uses a real January 2024 extract from the U.S. Bureau of Transportation Statistics to show how Jano can inspect a leakage-aware temporal plan, execute a multiclass model over those folds, and compare retraining policies with an ordinal cost.</p>
  <span class="jano-pill">walk-forward plan</span>
  <span class="jano-pill amber">multiclass target</span>
  <span class="jano-pill rose">retrain runner</span>
</div>
<p>Official source:</p>
<ul>
<li>BTS On-Time Performance overview: https://www.transtats.bts.gov/ontime/</li>
<li>BTS table download page: https://www.transtats.bts.gov/DL_SelectFields.aspx?QO_fu146_anzr=b0-gvzr&amp;gnoyr_VQ=FGJ</li>
<li>Direct January 2024 PREZIP file used here: https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip</li>
</ul>
<div class="jano-note jano-warning">The BTS zip/CSV is intentionally kept out of git because it is a large external dataset. The repository ignores <code>data/</code>, so this notebook is versioned but the downloaded dataset is not.</div>


In [1]:
from pathlib import Path
import sys
from urllib.request import urlretrieve
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "jano").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jano import (
    DriftBasedRetrain,
    DriftMonitoringPolicy,
    TemporalPartitionSpec,
    TemporalSemanticsSpec,
    TrainHistoryPolicy,
    WalkForwardPolicy,
    WalkForwardRunner,
)

BTS_ZIP_URL = 'https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip'
DATA_PATH = Path("data/bts/bts_ontime_2024_01.zip")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/bts/bts_ontime_2024_01.zip")

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(BTS_ZIP_URL, DATA_PATH)

STATE_LABELS = {0: "early", 1: "on_time", 2: "delayed"}


def hhmm_to_minutes(value):
    if pd.isna(value):
        return np.nan
    value = int(value)
    return (value // 100) * 60 + (value % 100)


def load_bts_sample(path: Path, limit: int = 25_000, origin: str = "ATL") -> pd.DataFrame:
    with zipfile.ZipFile(path) as zf:
        member = zf.namelist()[0]
        with zf.open(member) as f:
            frame = pd.read_csv(
                f,
                usecols=[
                    "FlightDate",
                    "DayOfWeek",
                    "Reporting_Airline",
                    "Origin",
                    "Dest",
                    "CRSDepTime",
                    "ArrTime",
                    "ArrDelay",
                    "Distance",
                    "Cancelled",
                    "Diverted",
                ],
            )
    frame = frame.loc[(frame["Origin"] == origin) & (frame["Cancelled"] == 0) & (frame["Diverted"] == 0)].copy()
    frame = frame.head(limit).copy()
    flight_date = pd.to_datetime(frame["FlightDate"])
    dep_minutes = frame["CRSDepTime"].map(hhmm_to_minutes)
    arr_minutes = frame["ArrTime"].map(hhmm_to_minutes)
    frame["scheduled_departure_at"] = flight_date + pd.to_timedelta(dep_minutes, unit="m")
    frame["actual_arrival_at"] = flight_date + pd.to_timedelta(arr_minutes, unit="m")
    overnight = frame["actual_arrival_at"] < frame["scheduled_departure_at"]
    frame.loc[overnight, "actual_arrival_at"] += pd.Timedelta(days=1)
    frame["scheduled_dep_hour"] = (dep_minutes // 60).astype("Int64")
    delay = frame["ArrDelay"].fillna(0)
    frame["arrival_state"] = np.select([delay <= -1, delay <= 15], [0, 1], default=2).astype(int)
    frame["arrival_state_label"] = frame["arrival_state"].map(STATE_LABELS)
    return frame.sort_values("scheduled_departure_at").reset_index(drop=True)


class SimpleArrivalStateClassifier:
    def fit(self, X: pd.DataFrame, y: pd.Series):
        keys = self._keys(X)
        grouped = pd.DataFrame({"key": keys, "target": y.to_numpy()}).groupby("key")["target"]
        self.mode_by_key = grouped.agg(lambda s: int(pd.Series.mode(s).iloc[0])).to_dict()
        self.global_mode = int(pd.Series(y).mode().iloc[0])
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        keys = self._keys(X)
        return np.array([self.mode_by_key.get(key, self.global_mode) for key in keys], dtype=int)

    @staticmethod
    def _keys(X: pd.DataFrame) -> pd.Series:
        return X[["Origin", "Dest", "Reporting_Airline", "DayOfWeek", "scheduled_dep_hour"]].astype(str).agg("|".join, axis=1)


def multiclass_accuracy(y_true, y_pred) -> float:
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def ordinal_delay_cost(y_true, y_pred) -> float:
    return float(np.mean(np.abs(np.asarray(y_true, dtype=int) - np.asarray(y_pred, dtype=int))))


## 1. Load a Real Operational Slice

The raw monthly BTS file is large enough to make planning realistic. We keep only ATL departures, remove cancelled/diverted flights and retain a compact set of columns so the notebook stays fast while preserving the temporal problem.


In [2]:
flights = load_bts_sample(DATA_PATH)
preview_columns = [
    "scheduled_departure_at",
    "actual_arrival_at",
    "Origin",
    "Dest",
    "Reporting_Airline",
    "ArrDelay",
    "arrival_state_label",
]
flights[preview_columns].head()


,scheduled_departure_at,actual_arrival_at,Origin,Dest,Reporting_Airline,ArrDelay,arrival_state_label
0,2024-01-01 05:21:00,2024-01-01 07:22:00,ATL,FLL,NK,7.0,on_time
1,2024-01-01 05:35:00,2024-01-01 07:36:00,ATL,LGA,NK,-24.0,early
2,2024-01-01 05:36:00,2024-01-01 07:13:00,ATL,MIA,AA,-20.0,early
3,2024-01-01 05:47:00,2024-01-01 07:34:00,ATL,DFW,AA,14.0,on_time
4,2024-01-01 05:50:00,2024-01-01 07:34:00,ATL,MCO,F9,2.0,on_time


## 2. Define the Multiclass Target

Instead of a binary delayed vs not-delayed target, this notebook uses three ordered arrival states:

- `early` when `ArrDelay <= -1`
- `on_time` when `-1 < ArrDelay <= 15`
- `delayed` when `ArrDelay > 15`

That makes fold-by-fold evaluation more interesting because we can inspect not only accuracy, but also an ordinal cost that penalizes confusing `early` with `delayed` more strongly than confusing adjacent states.


In [3]:
class_distribution = (
    flights["arrival_state_label"]
    .value_counts()
    .rename_axis("arrival_state_label")
    .reset_index(name="rows")
)
class_distribution["share"] = (class_distribution["rows"] / len(flights)).round(3)
class_distribution


,arrival_state_label,rows,share
0,early,15082,0.603
1,delayed,5023,0.201
2,on_time,4895,0.196


## 3. Define Leakage-Aware Time Semantics

The operational timeline is `scheduled_departure_at`. Training eligibility uses `actual_arrival_at`, because the supervised target is only known once the flight has arrived. Test eligibility stays anchored on scheduled departure.


In [4]:
semantics = TemporalSemanticsSpec(
    timeline_col="scheduled_departure_at",
    segment_time_cols={
        "train": "actual_arrival_at",
        "test": "scheduled_departure_at",
    },
)

walk_forward = WalkForwardPolicy(
    time_col=semantics,
    partition=TemporalPartitionSpec(
        layout="train_test",
        train_size="7D",
        test_size="1D",
        calendar_frequency="D",
    ),
    step="1D",
    strategy="rolling",
    max_folds=8,
)


## 4. Inspect the Plan Before Materializing Folds

A [`plan()`](https://marmurar.github.io/jano/concepts.html#planning-before-materialization) is the precomputed geometry of a simulation. It shows the temporal intent before any train/test slices are materialized or any model is trained.


In [5]:
plan = walk_forward.plan(flights, title="ATL January 2024 walk-forward plan")
plan_df = plan.to_frame()
plan_df[["iteration", "train_start", "train_end", "test_start", "test_end", "train_rows", "test_rows"]]


,iteration,train_start,train_end,test_start,test_end,train_rows,test_rows
0,0,2024-01-01,2024-01-08,2024-01-08,2024-01-09,5476,792
1,1,2024-01-02,2024-01-09,2024-01-09,2024-01-10,5594,751
2,2,2024-01-03,2024-01-10,2024-01-10,2024-01-11,5511,771
3,3,2024-01-04,2024-01-11,2024-01-11,2024-01-12,5480,806
4,4,2024-01-05,2024-01-12,2024-01-12,2024-01-13,5498,783
5,5,2024-01-06,2024-01-13,2024-01-13,2024-01-14,5461,724
6,6,2024-01-07,2024-01-14,2024-01-14,2024-01-15,5447,745
7,7,2024-01-08,2024-01-15,2024-01-15,2024-01-16,5378,801


## 5. Execute a Multiclass Model Across the Folds

[`WalkForwardRunner`](https://marmurar.github.io/jano/api.html#jano.runner.WalkForwardRunner) lets Jano own the training, prediction and metric loop while the splitter still owns temporal geometry. Here we compare four retraining policies over the same walk-forward [`plan`](https://marmurar.github.io/jano/concepts.html#planning-before-materialization):

- `always`: refit before every fold.
- `never`: fit once and keep using the same model.
- `periodic_2`: refit every two folds.
- `on_drift`: refit when the previous fold's `delay_cost` crosses the configured threshold.

The target has three ordered classes: `early`, `on_time` and `delayed`. The custom `delay_cost` metric is the mean absolute distance between those ordered classes, so predicting `early` for a delayed flight is penalized more than predicting `on_time`.


In [6]:
feature_cols = ["Origin", "Dest", "Reporting_Airline", "DayOfWeek", "scheduled_dep_hour", "Distance"]
metrics = {"accuracy": multiclass_accuracy, "delay_cost": ordinal_delay_cost}

policy_runs = {}
for label, kwargs in {
    "always": {"retrain": "always"},
    "never": {"retrain": "never"},
    "periodic_2": {"retrain": "periodic", "retrain_interval": 2},
    "on_drift": {
        "retrain_policy": DriftBasedRetrain(
            metric="delay_cost",
            threshold=0.15,
            baseline="last_retrain",
            relative=False,
        )
    },
}.items():
    policy_runs[label] = WalkForwardRunner(
        model=SimpleArrivalStateClassifier(),
        target_col="arrival_state",
        feature_cols=feature_cols,
        metrics=metrics,
        **kwargs,
    ).run(walk_forward, flights)

comparison = []
for label, run in policy_runs.items():
    fold_frame = run.to_frame()
    comparison.append(
        {
            "policy": label,
            "folds": len(fold_frame),
            "retrain_events": int(fold_frame["retrained"].sum()),
            "mean_accuracy": round(float(fold_frame["accuracy"].mean()), 3),
            "mean_delay_cost": round(float(fold_frame["delay_cost"].mean()), 3),
        }
    )
comparison_df = pd.DataFrame(comparison).sort_values(["mean_delay_cost", "mean_accuracy"], ascending=[True, False])
comparison_df


,policy,folds,retrain_events,mean_accuracy,mean_delay_cost
0,always,8,8,0.477,0.785
2,periodic_2,8,4,0.477,0.785
3,on_drift,8,3,0.477,0.785
1,never,8,1,0.475,0.793


In [7]:
periodic_run = policy_runs["periodic_2"]
periodic_report = periodic_run.report_data(include_predictions=False)
periodic_metric_trajectory = periodic_run.metric_trajectory()
periodic_retrain_events = periodic_run.retrain_events().merge(
    periodic_run.to_frame()[["fold", "accuracy", "delay_cost"]],
    on="fold",
    how="left",
)

print("report_data keys:", list(periodic_report.keys()))
print("This is the shape an AI agent or downstream dashboard can consume without parsing HTML.")

display(pd.DataFrame([periodic_report["summary"]]))
display(periodic_run.fold_summary())
display(periodic_metric_trajectory.head(12))
display(periodic_retrain_events)


report_data keys: ['summary', 'folds', 'metrics', 'retraining', 'metric_directions']
This is the shape an AI agent or downstream dashboard can consume without parsing HTML.


,folds,retrain_policy,retrain_events,retrain_ratio,metrics,accuracy_mean,accuracy_best,accuracy_best_fold,delay_cost_mean,delay_cost_best,delay_cost_best_fold
0,8,PeriodicRetrain,4,0.5,"[accuracy, delay_cost]",0.476782,0.625,0,0.785162,0.516213,2


,fold,retrained,last_retrain_fold,train_rows,test_rows,train_start,train_end,test_start,test_end
0,0,True,0,5476,792,2024-01-01,2024-01-08,2024-01-08,2024-01-09
1,1,False,0,5594,751,2024-01-02,2024-01-09,2024-01-09,2024-01-10
2,2,True,2,5511,771,2024-01-03,2024-01-10,2024-01-10,2024-01-11
3,3,False,2,5480,806,2024-01-04,2024-01-11,2024-01-11,2024-01-12
4,4,True,4,5498,783,2024-01-05,2024-01-12,2024-01-12,2024-01-13
5,5,False,4,5461,724,2024-01-06,2024-01-13,2024-01-13,2024-01-14
6,6,True,6,5447,745,2024-01-07,2024-01-14,2024-01-14,2024-01-15
7,7,False,6,5378,801,2024-01-08,2024-01-15,2024-01-15,2024-01-16


,fold,metric,value,direction,retrained
0,0,accuracy,0.625000,max,True
1,1,accuracy,0.199734,max,False
2,2,accuracy,0.616083,max,True
3,3,accuracy,0.596774,max,False
4,4,accuracy,0.403576,max,True
5,5,accuracy,0.453039,max,False
6,6,accuracy,0.499329,max,True
7,7,accuracy,0.420724,max,False
8,0,delay_cost,0.558081,min,True
9,1,delay_cost,1.383489,min,False


,fold,retrained,last_retrain_fold,train_rows,test_rows,train_start,train_end,test_start,test_end,accuracy,delay_cost
0,0,True,0,5476,792,2024-01-01,2024-01-08,2024-01-08,2024-01-09,0.625000,0.558081
1,2,True,2,5511,771,2024-01-03,2024-01-10,2024-01-10,2024-01-11,0.616083,0.516213
2,4,True,4,5498,783,2024-01-05,2024-01-12,2024-01-12,2024-01-13,0.403576,0.908046
3,6,True,6,5447,745,2024-01-07,2024-01-14,2024-01-14,2024-01-15,0.499329,0.719463


## 6. Train History Hypothesis with the Same Multiclass Target

Now the question changes: if the test day stays fixed, how much train history is actually needed to recover the best cost profile?


In [8]:
train_history = TrainHistoryPolicy(
    semantics,
    cutoff=plan.to_frame().iloc[3]["train_end"],
    train_sizes=["2D", "4D", "6D", "7D"],
    test_size="1D",
)

train_history_result = train_history.evaluate(
    flights,
    model=SimpleArrivalStateClassifier(),
    target_col="arrival_state",
    feature_cols=feature_cols,
    metrics=metrics,
)

train_history_result.to_frame()[["train_size", "train_rows", "accuracy", "delay_cost"]]


,train_size,train_rows,accuracy,delay_cost
0,2 days 00:00:00,1530,0.125310,1.512407
1,4 days 00:00:00,3129,0.637717,0.487593
2,6 days 00:00:00,4685,0.633995,0.493797
3,7 days 00:00:00,5480,0.596774,0.524814


## 7. Drift Monitoring with Ordinal Cost

This policy freezes the train window and moves test forward. The metric to watch is now more operational: accuracy may stay similar while ordinal delay cost worsens because the classifier drifts toward more severe confusions.


In [9]:
drift_monitor = DriftMonitoringPolicy(
    semantics,
    cutoff=plan.to_frame().iloc[3]["train_end"],
    train_size="7D",
    test_size="1D",
    step="1D",
    max_windows=6,
)

drift_result = drift_monitor.evaluate(
    flights,
    model=SimpleArrivalStateClassifier(),
    target_col="arrival_state",
    feature_cols=feature_cols,
    metrics=metrics,
)

drift_result.to_frame()[["window", "accuracy", "delay_cost"]]


,window,accuracy,delay_cost
0,0,0.596774,0.524814
1,1,0.403576,0.908046
2,2,0.453039,0.787293
3,3,0.499329,0.719463
4,4,0.420724,0.883895
5,5,0.417500,0.885000


## Notes

- The notebook still prioritizes temporal geometry and leakage-aware validation over model sophistication.
- `TemporalBacktestSplitter` remains available when you want manual fold iteration; the runner is an execution layer on top of that geometry.
- The multiclass target makes fold variation more informative because `delay_cost` distinguishes adjacent mistakes from severe ones.
- Replace the toy classifier with your own model pipeline while keeping the same temporal setup.
